# 类别不平衡如何治理？

**面试回答：**先定义漏报/误报成本和审核预算，再用类权重、阈值与 PR/recall@K；accuracy 在稀疏正类下会误导。

## 真实案例

12 笔支付中只有 3 笔欺诈，模型已有风险分数。

In [1]:
import numpy as np  # 导入 NumPy 计算不平衡指标。
order=np.array([f'F{i:02d}' for i in range(1,13)])  # 构造订单编号。
y=np.array([0,0,0,0,1,0,0,1,0,0,0,1])  # 标记欺诈正类。
p=np.array([.02,.04,.08,.12,.61,.15,.2,.72,.18,.29,.35,.52])  # 记录风险概率。
print('订单/标签/风险=',list(zip(order.tolist(),y.tolist(),p.tolist())))  # 输出业务账本。
print('欺诈先验=',round(float(y.mean()),3))  # 输出正类比例。

订单/标签/风险= [('F01', 0, 0.02), ('F02', 0, 0.04), ('F03', 0, 0.08), ('F04', 0, 0.12), ('F05', 1, 0.61), ('F06', 0, 0.15), ('F07', 0, 0.2), ('F08', 1, 0.72), ('F09', 0, 0.18), ('F10', 0, 0.29), ('F11', 0, 0.35), ('F12', 1, 0.52)]
欺诈先验= 0.25


## Baseline / 基线

全预测正常有很高 accuracy，却没有欺诈召回。

In [2]:
base=np.zeros(len(y),dtype=int)  # 构造全正常基线。
base_acc=float(np.mean(base==y))  # 计算表面准确率。
print('全正常 accuracy=',round(base_acc,3),'recall=0')  # 输出误导性基线。
print('正类总数=',int(y.sum()))  # 输出必须召回的欺诈数。

全正常 accuracy= 0.75 recall=0
正类总数= 3


In [3]:
def report(threshold):  # 定义阈值下的 precision/recall。
    pred=p>=threshold  # 生成审核决策。
    tp=int(np.sum(pred&(y==1)))  # 统计真阳性。
    fp=int(np.sum(pred&(y==0)))  # 统计假阳性。
    return tp/(tp+fp) if tp+fp else 1.0,tp/int(y.sum()),int(pred.sum())  # 返回精确率、召回率和审核量。
for threshold in [.5,.3,.15]:  # 比较不同阈值。
    print('阈值',threshold,'precision/recall/量=',tuple(round(v,3) if isinstance(v,float) else v for v in report(threshold)))  # 输出业务取舍。
class_weight=(len(y)-y.sum())/y.sum()  # 计算正类平衡权重。
weighted_loss=-np.mean(class_weight*y*np.log(p)+(1-y)*np.log(1-p))  # 手写加权交叉熵示例。
print('正类权重/加权损失=',round(float(class_weight),2),round(float(weighted_loss),3))  # 输出训练中间量。

阈值 0.5 precision/recall/量= (1.0, 1.0, 3)
阈值 0.3 precision/recall/量= (0.75, 1.0, 4)
阈值 0.15 precision/recall/量= (0.375, 1.0, 8)
正类权重/加权损失= 3.0 0.505


## 结果解读

类权重改变训练梯度，阈值改变上线决策；二者不同且都可能影响校准。

In [4]:
precision,recall,count=report(.3)  # 选择与人工预算匹配的示例阈值。
print('阈值0.3审核订单=',order[p>=.3].tolist())  # 输出审核队列。
print('阈值0.3 precision/recall=',round(precision,3),round(recall,3))  # 输出主要结果。
print('生产差距：需时间回放、金额成本、分组公平性、校准与先验漂移监控。')  # 说明边界。

阈值0.3审核订单= ['F05', 'F08', 'F11', 'F12']
阈值0.3 precision/recall= 0.75 1.0
生产差距：需时间回放、金额成本、分组公平性、校准与先验漂移监控。


## 失败案例与修复

只优化 accuracy 会选择全正常；修复是以 recall@预算/PR 和成本选择阈值。

In [5]:
print('失败全正常 accuracy=',base_acc,'recall=0')  # 输出失败证据。
print('修复阈值0.3 recall=',round(recall,3),'审核量=',count)  # 输出修复结果。
print('过采样只能在训练折进行，不能复制到验证集。')  # 说明泄漏护栏。
print('高 recall 不代表低业务损失。')  # 说明成本边界。

失败全正常 accuracy= 0.75 recall=0
修复阈值0.3 recall= 1.0 审核量= 4
过采样只能在训练折进行，不能复制到验证集。
高 recall 不代表低业务损失。


In [6]:
assert len(order)>=5  # 保护样本数。
assert base_acc>.5  # 保护高 accuracy 反例。
assert recall>0  # 保护阈值召回欺诈。
assert class_weight>1  # 保护正类重加权。